# Amazon Bedrock AgentCore Memory의 Cross-Region Replication

## 개요

이 튜토리얼에서는 [Memory Record Streaming](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/memory-record-streaming.html) 기능을 사용하여 Amazon Bedrock AgentCore Memory의 **active-passive cross-region replication**을 구축하는 방법을 살펴봅니다. Primary 리전에서 Memory Record event를 streaming하면 Lambda consumer가 거의 실시간으로 Secondary 리전에 복제합니다.

### 튜토리얼 세부 정보

| 정보 | 세부 정보 |
|:-----------|:--------|
| 튜토리얼 유형 | Cross-Region Replication |
| 기능 | Memory Record Streaming + Lambda Consumer |
| 주요 기능 | Kinesis, Lambda ESM, CloudFormation, DynamoDB Global Table |
| 예제 난이도 | 고급 |
| 사용 SDK | boto3, AWS CLI |

### 학습 내용

1. CloudFormation으로 두 리전에 복제 인프라 배포
2. Streaming이 활성화된 AgentCore Memory 인스턴스 생성
3. Record가 Primary에서 Secondary로 거의 실시간 복제되는지 검증
4. 리전 간 Streaming을 전환하여 failover 수행
5. 모든 리소스 정리

### 아키텍처

```mermaid
flowchart LR
    subgraph primary["Primary (us-east-1)"]
        PM[AgentCore Memory<br/>streaming: ON]
        PK[Kinesis Stream]
        PL[Lambda Consumer]
        PDLQ[SQS DLQ]
    end

    subgraph secondary["Secondary (us-west-2)"]
        SM[AgentCore Memory<br/>streaming: OFF]
        SK[Kinesis Stream]
        SL["Lambda (idle)"]
    end

    DDB[(DynamoDB Global Table<br/>ACTIVE_REGION)]

    PM -->|stream events| PK
    PK -->|ESM trigger| PL
    PL -->|BatchCreateMemoryRecords| SM
    PL -.->|on failure| PDLQ
    SK -.->|no data flowing| SL

    style primary fill:#e8f5e9,stroke:#2e7d32
    style secondary fill:#fff3e0,stroke:#ef6c00
    style PM fill:#c8e6c9
    style SM fill:#ffe0b2
```

### 복제 흐름

```mermaid
sequenceDiagram
    participant App as Application
    participant PM as Primary Memory
    participant KS as Kinesis Stream
    participant LC as Lambda Consumer
    participant SM as Secondary Memory

    App->>PM: BatchCreateMemoryRecords
    PM->>KS: MemoryRecordCreated event
    KS->>LC: ESM triggers Lambda
    LC->>LC: Check loop prevention (replicated/ prefix)
    LC->>SM: BatchCreateMemoryRecords (ns: replicated/...)
    Note over SM: Record stored with<br/>replicated/ namespace
```

### Failover 흐름

```mermaid
sequenceDiagram
    participant Op as Operator
    participant SM as Secondary Memory
    participant PM as Primary Memory
    participant DDB as DynamoDB

    Op->>SM: update-memory (enable streaming)
    Op->>PM: update-memory (disable streaming)
    Op->>DDB: Set ACTIVE_REGION = us-west-2
    Note over SM,PM: Secondary is now active,<br/>replicating to primary
```

**주요 설계 결정:**
- Secondary에서는 Streaming을 **OFF**로 유지하여 loopback event 비용 제거
- 두 리전 모두 Lambda ESM을 활성화 상태로 유지하며 Streaming이 꺼져 있으면 유휴 상태로 대기
- Failover = Streaming 전환(API 호출 2회, 수초 소요)
- `replicated/` namespace prefix로 loop 방지

### 사전 요구 사항

- **Python 3.10 이상**
- **AWS CLI v2.34 이상** - AgentCore Memory API에는 최신 CLI 버전이 필요합니다. `brew upgrade awscli`(macOS) 또는 `pip install --upgrade awscli`로 업데이트하세요.
- **boto3 >= 1.42.63** - 첫 번째 셀에서 자동 설치
- `us-east-1` 및 `us-west-2` 모두에서 **Amazon Bedrock AgentCore** 액세스 활성화

### 필요한 IAM 권한

AWS 자격 증명에는 다음 서비스의 권한이 필요합니다.

| 서비스 | 주요 Action | 용도 |
|:--------|:-----------|:----|
| **Bedrock AgentCore** | `CreateMemory`, `DeleteMemory`, `GetMemory`, `ListMemories`, `UpdateMemory`, `BatchCreateMemoryRecords`, `ListMemoryRecords` | Memory 인스턴스 생성/관리, record 읽기/쓰기, Streaming 구성 |
| **CloudFormation** | `CreateStack`, `UpdateStack`, `DeleteStack`, `DescribeStacks`, `CreateChangeSet`, `ExecuteChangeSet` | 인프라 stack 배포 및 삭제 |
| **Kinesis** | `CreateStream`, `DeleteStream`, `DescribeStream`, `GetShardIterator`, `GetRecords` | Stream 생성, event 읽기(4단계) |
| **Lambda** | `CreateFunction`, `UpdateFunctionCode`, `UpdateFunctionConfiguration`, `DeleteFunction`, `CreateEventSourceMapping`, `DeleteEventSourceMapping` | 복제 consumer 및 ESM 배포 |
| **IAM** | `CreateRole`, `PutRolePolicy`, `PassRole`, `DeleteRole`, `DeleteRolePolicy` | Lambda 및 Memory Streaming용 실행 역할 생성 |
| **SQS** | `CreateQueue`, `DeleteQueue`, `SendMessage`, `GetQueueAttributes` | Dead Letter Queue 생성/관리 |
| **DynamoDB** | `CreateTable`, `DeleteTable`, `PutItem`, `GetItem`, `DescribeTable` | Global Table 생성, 활성 리전 구성 읽기/쓰기 |
| **S3** | `CreateBucket`, `PutObject`, `GetObject`, `DeleteObject`, `DeleteBucket`, `ListBucket` | Lambda 배포 package 저장 |
| **STS** | `GetCallerIdentity` | 계정 ID 확인 |
| **CloudWatch** | `PutMetricAlarm`, `DeleteAlarms`, `PutDashboard`, `DeleteDashboards` | Monitoring alarm 및 dashboard 생성 |

## 0단계: 환경 설정

필요한 SDK를 설치하고 두 대상 리전을 구성하여 시작합니다. 이 Notebook은 기본적으로 `us-east-1`을 Primary로, `us-west-2`를 Secondary로 사용하며 환경 변수 `PRIMARY_REGION`과 `SECONDARY_REGION`으로 재정의할 수 있습니다.

In [ ]:
!pip install -qr requirements.txt

In [ ]:
import os
import json
import time
import base64
import logging
import boto3
from datetime import datetime, timezone

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

# 리전 구성 - 필요하면 환경 변수로 재정의
PRIMARY_REGION = os.getenv("PRIMARY_REGION", "us-east-1")
SECONDARY_REGION = os.getenv("SECONDARY_REGION", "us-west-2")
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]

# 리소스 정리 셀에서 CloudFormation stack 및 S3 bucket 검색에 사용
STACK_PREFIX = "agentcore-replication"

print(f"Account: {ACCOUNT_ID}")
print(f"Primary: {PRIMARY_REGION}  Secondary: {SECONDARY_REGION}")

## 1단계: 인프라 배포

`scripts/deploy.sh` script는 두 리전의 전체 배포를 조율합니다. 수행 작업은 다음과 같습니다.

1. **Lambda packaging** - `scripts/handler.py`와 종속성을 zip으로 묶어 두 리전의 S3에 업로드
2. **DynamoDB Global Table 배포** - 현재 활성 리전을 추적하기 위해 두 리전에 복제되는 단일 record table(`scripts/global-stack.yaml`) 배포
3. **리전별 stack 배포** - 각 리전에 Kinesis Data Stream, Lambda consumer, SQS Dead Letter Queue, IAM 역할, CloudWatch alarm 배포(`scripts/regional-stack.yaml`)
4. **AgentCore Memory 인스턴스 생성** - Primary는 Streaming ON(event가 Kinesis로 흐름), Secondary는 Streaming 없이 생성(인프라는 준비되지만 유휴 상태)
5. **Cross-region ID 연결** - 복제 대상을 알 수 있도록 각 리전의 Lambda 환경 변수를 원격 Memory ID로 업데이트
6. **초기 구성 입력** - DynamoDB에 `ACTIVE_REGION = us-east-1` 기록

약 5분이 소요됩니다. 아래에서 각 단계의 진행 출력을 확인할 수 있습니다.

In [ ]:
!bash scripts/deploy.sh {PRIMARY_REGION} {SECONDARY_REGION}

## 2단계: 배포 검증

`deploy.sh` script는 두 Memory ID를 DynamoDB config table에 저장합니다. 여기서는 이를 다시 읽어 두 Memory가 모두 `ACTIVE` 상태인지 확인합니다.

여기서 가져온 Memory ID(`primary_memory_id` 및 `secondary_memory_id`)는 Notebook의 나머지 부분에서 사용됩니다.


In [ ]:
def get_memory_id(key):
    """DynamoDB 구성 테이블에서 메모리 ID를 조회합니다."""
    ddb = boto3.client("dynamodb", region_name=PRIMARY_REGION)
    item = ddb.get_item(TableName="AgentCoreMemoryReplicationConfig", Key={"PK": {"S": key}}).get("Item", {})
    return item.get("memory_id", {}).get("S")


# 배포 script가 config table에 Memory ID 저장
print("Reading memory IDs from config table...")
primary_memory_id = get_memory_id("MEMORY_ID_PRIMARY")
secondary_memory_id = get_memory_id("MEMORY_ID_SECONDARY")

# 두 Memory가 모두 ACTIVE 상태인지 검증
for label, mid, region in [
    ("Primary", primary_memory_id, PRIMARY_REGION),
    ("Secondary", secondary_memory_id, SECONDARY_REGION),
]:
    if mid:
        client = boto3.client("bedrock-agentcore-control", region_name=region)
        status = client.get_memory(memoryId=mid)["memory"]["status"]
        print(f"{label} Memory: {mid} ({status})")
    else:
        print(f"{label} Memory: NOT FOUND")

assert primary_memory_id and secondary_memory_id, "Deployment incomplete — run scripts/deploy.sh first"

In [ ]:
# 활성 리전 추적 검증
ddb = boto3.client("dynamodb", region_name=PRIMARY_REGION)
item = ddb.get_item(TableName="AgentCoreMemoryReplicationConfig", Key={"PK": {"S": "ACTIVE_REGION"}}).get("Item", {})

print(f"Active region: {item.get('region', {}).get('S', 'NOT SET')}")

## 3단계: 복제 테스트

핵심 테스트입니다. **Primary** 리전에 Memory Record를 생성하고 **Secondary** 리전에 자동으로 나타나는지 검증합니다.

복제 pipeline은 다음과 같이 작동합니다.
1. Primary Memory에서 `BatchCreateMemoryRecords` 호출
2. Streaming이 ON이므로 각 record가 Primary의 Kinesis stream에서 `MemoryRecordCreated` event trigger
3. Event Source Mapping으로 연결된 Lambda consumer가 event 수신
4. Lambda가 loop 방지를 위해 namespace에 `replicated/` prefix를 붙여 Secondary Memory에서 `BatchCreateMemoryRecords` 호출
5. Record가 Secondary의 `replicated/` namespace에 나타남

### 3a. Primary에 테스트 record 생성

실제 Agent Memory의 사용자 선호도와 평가를 모의하기 위해 서로 다른 namespace에 record 3개를 생성합니다.

In [ ]:
# Primary 리전의 AgentCore Memory data plane client 생성
primary_client = boto3.client("bedrock-agentcore", region_name=PRIMARY_REGION)

# 실제 Agent Memory를 모의하는 테스트 record - 선호도 및 평가
test_records = [
    {"text": "User prefers Python for backend services", "ns": "user/alice"},
    {"text": "User likes event-driven architectures with Lambda", "ns": "user/alice"},
    {"text": "User is evaluating multi-region disaster recovery", "ns": "user/bob"},
]

created_ids = []
for i, rec in enumerate(test_records):
    resp = primary_client.batch_create_memory_records(
        memoryId=primary_memory_id,
        records=[
            {
                "requestIdentifier": f"test-{i}-{int(time.time())}",  # Idempotency를 위한 고유 ID
                "content": {"text": rec["text"]},
                "namespaces": [rec["ns"]],  # 예: user/alice
                "timestamp": str(int(time.time())),  # 문자열 형식의 epoch 초
            }
        ],
    )
    rid = resp["successfulRecords"][0]["memoryRecordId"]
    created_ids.append(rid)
    print(f"Created: {rid} — {rec['text'][:60]}")

print(f"\n✅ Created {len(created_ids)} records in {PRIMARY_REGION}")

### 3b. 복제 대기 및 Secondary에서 검증

복제 pipeline(Memory → Kinesis → Lambda → 원격 Memory)은 일반적으로 end-to-end 10~30초가 걸립니다. Record 3개가 모두 나타날 때까지 Secondary의 `replicated/` namespace를 polling합니다.

복제가 작동하면 Primary와 동일한 record 텍스트가 `replicated/` prefix가 붙은 namespace(예: `replicated/user/alice`)로 Secondary에 저장된 것을 확인할 수 있습니다.

In [ ]:
# Secondary 리전의 AgentCore Memory data plane client 생성
secondary_client = boto3.client("bedrock-agentcore", region_name=SECONDARY_REGION)

# Secondary의 'replicated/' namespace에서 record polling
# Lambda consumer는 원격 리전에 쓸 때 namespace에 'replicated/' prefix 추가
# 따라서 Primary의 'user/alice'는 Secondary에서 'replicated/user/alice'가 됨
print("Waiting for replication (polling every 10s, up to 120s)...\n")
start = time.time()
replicated = []

while time.time() - start < 120:
    try:
        resp = secondary_client.list_memory_records(
            memoryId=secondary_memory_id,
            namespacePath="replicated/",  # 계층적 일치 - replicated/user/alice, replicated/user/bob 등을 검색
        )
        replicated = resp.get("memoryRecordSummaries", [])
        if len(replicated) >= len(test_records):
            break
    except Exception:
        pass
    time.sleep(10)

elapsed = time.time() - start
print(f"Found {len(replicated)} replicated record(s) in {elapsed:.0f}s:\n")
for r in replicated:
    text = r.get("content", {}).get("text", "N/A")[:80]
    print(f"  {r['memoryRecordId']} — {text}")

if len(replicated) >= len(test_records):
    print(f"\n✅ All {len(test_records)} records replicated successfully!")
else:
    print(f"\n⚠️  Only {len(replicated)}/{len(test_records)} replicated. Check Lambda logs for errors.")

## 4단계: Kinesis Stream Event 읽기

내부 동작을 확인하기 위해 Primary의 Kinesis stream을 직접 읽습니다. 이는 record가 변경될 때 AgentCore Memory가 게시하고 Lambda consumer가 처리하는 것과 동일한 원시 event입니다.

다음 event가 표시되어야 합니다.
- Streaming을 처음 구성할 때 게시된 `StreamingEnabled` event
- 3단계에서 생성한 각 record의 `MemoryRecordCreated` event

각 event에는 event 유형, Memory Record ID, namespace, 실제 record 텍스트(`FULL_CONTENT` 수준)가 포함됩니다.

In [ ]:
# Primary의 Kinesis stream에서 원시 event 읽기
# Lambda consumer가 처리하는 것과 동일한 event
kinesis = boto3.client("kinesis", region_name=PRIMARY_REGION)
stream_info = kinesis.describe_stream(StreamName="agentcore-memory-stream")
shard_id = stream_info["StreamDescription"]["Shards"][0]["ShardId"]

# 가장 오래된 record부터 시작(TRIM_HORIZON)
iterator = kinesis.get_shard_iterator(
    StreamName="agentcore-memory-stream",
    ShardId=shard_id,
    ShardIteratorType="TRIM_HORIZON",
)["ShardIterator"]

events = []
for _ in range(5):  # 최대 5회 polling
    resp = kinesis.get_records(ShardIterator=iterator, Limit=100)
    for rec in resp["Records"]:
        data = base64.b64decode(rec["Data"]) if isinstance(rec["Data"], str) else rec["Data"]
        events.append(json.loads(data))
    iterator = resp["NextShardIterator"]
    if not resp["Records"]:
        time.sleep(2)

print(f"Read {len(events)} event(s) from Kinesis:\n")
for i, evt in enumerate(events[:10]):
    se = evt.get("memoryStreamEvent", {})
    print(f"  [{se.get('eventType', '?')}] {se.get('memoryRecordId', 'N/A')[:40]}  ns={se.get('namespaces', [])}")

## 5단계: Failover 테스트

이제 활성 리전을 Primary에서 Secondary로 전환하여 리전 장애를 모의합니다. Failover 과정은 다음과 같습니다.

1. **Secondary에서 Streaming 활성화** - Secondary의 Kinesis stream이 event 수신을 시작하고 Lambda가 Primary로 복제 시작
2. **Primary에서 Streaming 비활성화** - Primary가 event 게시 중지
3. **DynamoDB 업데이트** - 애플리케이션 계층이 대상을 알 수 있도록 `ACTIVE_REGION`을 Secondary로 설정

순서가 중요합니다. 기존 path를 비활성화하기 **전에** 새 path를 활성화합니다. 이렇게 하면 복제 공백이 발생하지 않습니다. 잠시 두 리전의 Streaming이 모두 켜져 있어도 `replicated/` namespace prefix가 무한 loop를 방지합니다.

### 5a. Streaming 전환

In [ ]:
# Secondary를 먼저 활성화 - 역방향 복제 path 시작
# 정방향 path를 끊기 전에 수행하여 복제 공백 방지
!bash scripts/toggle-streaming.sh enable {SECONDARY_REGION}

# 그런 다음 Primary를 비활성화하여 Kinesis event 게시 중지
!bash scripts/toggle-streaming.sh disable {PRIMARY_REGION}

In [ ]:
# DynamoDB의 활성 리전 업데이트
ddb.put_item(
    TableName="AgentCoreMemoryReplicationConfig",
    Item={
        "PK": {"S": "ACTIVE_REGION"},
        "region": {"S": SECONDARY_REGION},
        "updated_at": {"S": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")},
        "updated_by": {"S": "notebook-failover"},
    },
)
print(f"Active region updated to: {SECONDARY_REGION}")

### 5b. Failover 검증 - Secondary에 쓰고 Primary 복제 확인

Secondary가 활성화되었으므로 복제가 역방향으로 작동하는지 확인합니다. Secondary에 record를 생성하고 Primary의 `replicated/` namespace에 나타나는지 검증합니다.

이를 통해 전체 failover를 확인할 수 있습니다. 이제 Secondary가 source of truth이고 Primary가 복제된 데이터를 수신합니다.

In [ ]:
# 현재 활성 리전인 Secondary에 record 쓰기
resp = secondary_client.batch_create_memory_records(
    memoryId=secondary_memory_id,
    records=[
        {
            "requestIdentifier": f"failover-test-{int(time.time())}",
            "content": {"text": "Record created during failover in secondary region"},
            "namespaces": ["user/failover-test"],
            "timestamp": str(int(time.time())),
        }
    ],
)
failover_id = resp["successfulRecords"][0]["memoryRecordId"]
print(f"Created in secondary: {failover_id}")

# Primary에서 복제된 record polling
# 이제 Secondary의 Lambda가 Primary로 복제
print("Waiting for replication to primary (polling every 10s, up to 120s)...\n")
start = time.time()
recs = []
while time.time() - start < 120:
    try:
        recs = primary_client.list_memory_records(memoryId=primary_memory_id, namespacePath="replicated/").get(
            "memoryRecordSummaries", []
        )
        if recs:
            break
    except Exception:
        pass
    time.sleep(10)

if recs:
    for r in recs:
        print(f"  {r['memoryRecordId']} — {r.get('content', {}).get('text', '')[:80]}")
    print(f"\n✅ Failover replication working! ({time.time() - start:.0f}s)")
else:
    print("⚠️  No replicated records yet — check Lambda logs in secondary region")

### 5c. Failback - Primary를 활성 상태로 복원

Failback은 동일한 과정을 역순으로 수행합니다. Primary Streaming을 활성화하고 Secondary를 비활성화한 뒤 DynamoDB를 업데이트합니다. 이후 시스템은 원래 구성으로 돌아갑니다.

In [ ]:
# Failback: 원래 구성 복원
# 동일한 과정을 역순으로 수행 - Primary 활성화, Secondary 비활성화
!bash scripts/toggle-streaming.sh enable {PRIMARY_REGION}
!bash scripts/toggle-streaming.sh disable {SECONDARY_REGION}

# 애플리케이션 계층이 Primary가 다시 활성화되었음을 알 수 있도록 config table 업데이트
ddb.put_item(
    TableName="AgentCoreMemoryReplicationConfig",
    Item={
        "PK": {"S": "ACTIVE_REGION"},
        "region": {"S": PRIMARY_REGION},
        "updated_at": {"S": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")},
        "updated_by": {"S": "notebook-failback"},
    },
)
print(f"\n✅ Failback complete. Active region: {PRIMARY_REGION}")

## 6단계: 리소스 정리

이 튜토리얼에서 생성한 모든 리소스를 삭제합니다. 삭제 대상은 다음과 같습니다.
- 두 리전의 AgentCore Memory 인스턴스(삭제 완료까지 대기)
- 두 리전의 CloudFormation stack(Kinesis, Lambda, SQS, IAM, CloudWatch)
- DynamoDB Global Table
- Lambda 배포 package에 사용된 S3 bucket

> **비용 참고:** Kinesis Data Streams에는 shard당 시간 요금(각각 월 약 11달러)이 부과됩니다. 지속적인 비용을 방지하려면 실습을 마친 후 리소스 정리를 실행하세요.

In [ ]:
# AgentCore Memory 인스턴스 삭제
for region, mid in [
    (PRIMARY_REGION, primary_memory_id),
    (SECONDARY_REGION, secondary_memory_id),
]:
    try:
        client = boto3.client("bedrock-agentcore-control", region_name=region)
        client.delete_memory(memoryId=mid)
        print(f"Deleting memory {mid} in {region}...")
        # 삭제 대기
        for _ in range(30):
            try:
                status = client.get_memory(memoryId=mid)["memory"]["status"]
                if status == "DELETING":
                    time.sleep(5)
                else:
                    break
            except client.exceptions.ResourceNotFoundException:
                print("  ✅ Deleted")
                break
    except Exception as e:
        print(f"  Error: {e}")

In [ ]:
# CloudFormation stack 삭제
for region in [PRIMARY_REGION, SECONDARY_REGION]:
    cf = boto3.client("cloudformation", region_name=region)
    try:
        cf.delete_stack(StackName=f"{STACK_PREFIX}-regional")
        print(f"Deleting regional stack in {region}...")
    except Exception as e:
        print(f"  Error: {e}")

# Global stack(Primary에만 존재)
try:
    cf_primary = boto3.client("cloudformation", region_name=PRIMARY_REGION)
    cf_primary.delete_stack(StackName=f"{STACK_PREFIX}-global")
    print("Deleting global stack...")
except Exception as e:
    print(f"  Error: {e}")

# S3 bucket 정리
for region in [PRIMARY_REGION, SECONDARY_REGION]:
    bucket = f"{STACK_PREFIX}-artifacts-{ACCOUNT_ID}-{region}"
    try:
        s3 = boto3.client("s3", region_name=region)
        objs = s3.list_objects_v2(Bucket=bucket).get("Contents", [])
        for obj in objs:
            s3.delete_object(Bucket=bucket, Key=obj["Key"])
        s3.delete_bucket(Bucket=bucket)
        print(f"Deleted S3 bucket: {bucket}")
    except Exception as e:
        print(f"  S3 cleanup ({bucket}): {e}")

print("\n✅ Cleanup complete")

## 마무리

이 튜토리얼에서는 AgentCore Memory의 end-to-end cross-region replication을 구축했습니다.

1. **인프라 배포** - 두 리전에 Kinesis stream, Lambda consumer, SQS DLQ, IAM 역할, CloudWatch alarm 배포
2. **Memory 인스턴스 생성** - Primary는 Streaming ON, Secondary는 Streaming OFF
3. **복제 검증** - Primary에서 생성한 record가 Secondary의 `replicated/` namespace에 나타나는지 확인
4. **Failover/Failback 테스트** - 리전 간 Streaming을 수초 내 전환

### 핵심 요점

| 지표 | 값 |
|:-------|:------|
| RPO(Recovery Point Objective) | 5~15초 |
| RTO(Recovery Time Objective) | 15~30초 |
| Failover 메커니즘 | `update-memory` API를 통해 Streaming 전환 |
| Loop 방지 | `replicated/` namespace prefix |
| 충돌 해결 | AgentCore Memory 기본 통합 기능 |

### 다음 단계

- 자동 failover 감지를 위한 CloudWatch alarm 추가
- DNS 수준 failover를 위해 Route 53 health check와 통합
- 3개 이상 리전 topology로 확장
- Cross-account replication 지원 추가
